<a href="https://colab.research.google.com/github/JuanAntonioLeonOjeda/03MIAR---Algoritmos-de-Optimizacion/blob/main/Trabajo_Pr%C3%A1ctico_Juan_Antonio_Le%C3%B3n_Ojeda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Juan Antonio León Ojeda  <br>
Url: https://github.com/JuanAntonioLeonOjeda/03MIAR---Algoritmos-de-Optimizacion/tree/main/TRABAJO_PRACTICO<br>
Google Colab: https://colab.research.google.com/drive/17i9e51u3h6eMk9lQVqFDdGxHE9kPpins <br>
Problema:
**3. Configuración de Tribunales**

Descripción del problema:

Se precisa configurar tribunales de evaluación para un grupo de 15 alumnos que desean
presentar su Trabajo Fin de Máster (TFM).
Cada tribunal está compuesto por tres profesores, cada uno desempeñando uno de los
siguientes roles: Presidente, Secretario o Vocal.
Los profesores han indicado su disponibilidad horaria para participar en los tribunales de
15h a 21h durante la semana del 15 al 19 de abril: <br>
Número de profesores : 10 <br>
Número de tribunales : 15 <br>
Hay 15 alumnos, por lo que se deben configurar 15 tribunales buscando la configuración
más equilibrada posible en cuanto a la cantidad de tribunales asignados a cada profesor, es
decir, evitando que un profesor tenga muchos tribunales y otros pocos.
Obviamente ningún profesor puede asistir a dos tribunales a la misma fecha/hora y no
puede ser convocado a un tribunal al que no tiene disponibilidad.









                                        

#Modelo

- ¿Como represento el espacio de soluciones?

In [20]:
# El espacio de soluciones será un array de diccionarios (podría usarse simplemente arrays anidados, pero los diccionarios proporcionarán una mayor claridad de cada asignación).
# El formato será como el que sigue:
'''
solucion = [
    {
        tibunal: 0,
        dia: 15,
        hora: 16,
        p: 'ABC',  # presidente
        s: 'DEF',  # secretario
        v: 'GHI'.  # vocal
    },
    ...
]
'''

"\nsolucion = [\n    {\n        tibunal: 0,\n        dia: 15,\n        hora: 16,\n        p: 'ABC',  # presidente\n        s: 'DEF',  # secretario\n        v: 'GHI'.  # vocal\n    },\n    ...\n]\n"

- ¿Cual es la función objetivo?

El objetivo principal del modelo tiene que ser alcanzar un reparto equilibrado de asignaciones de tribunales para cada profesor.
Para ello, se calculará la desviación estándar para medir la dispersión de los valores (cantidad de asignaciones) respecto a la media.
La función objetivo tiene que intentar que esta desviación estándar sea la menor posible para cada profesor, a través de la siguiente fórmula

σ = √[Σ(xi - x̄)² / n]

Donde:
xi: número de tribulaes asignados al profesor i
x̄: media de asignaciones
n: total de profesores

- ¿Como implemento las restricciones?

In [21]:
# Las restricciones serán un conjunto de arrays y de diccionarios. Básicamente se usarán arrays para listar los elementos por los que hay que iterar (profesores, días y horas) y diccionarios para hacer consultas (disponibilidad y roles de cada profesro)

profesores = ['RRD', 'QYV', 'LHL', 'HLC', 'MSB', 'PMQ', 'QWF', 'EBB', 'IOE', 'IOA']
dias = [15, 16, 17, 18, 19]
horas = [15, 16, 17, 18, 19, 20, 21]
disponibilidad = {
    'RRD': { # Revisado
        (15, 15): 0, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 0, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 0, (17, 18): 0, (17, 19): 1, (17, 20): 0, (17, 21): 1,
        (18, 15): 0, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 0, (19, 21): 0,
    },
    'QYV': { # Revisado
        (15, 15): 1, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 0, (15, 21): 0,
        (16, 15): 0, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 0, (16, 21): 0,
        (17, 15): 1, (17, 16): 0, (17, 17): 0, (17, 18): 1, (17, 19): 1, (17, 20): 1, (17, 21): 0,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 1,
    },
    'LHL': { #Revisado
        (15, 15): 0, (15, 16): 0, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 1, (16, 17): 1, (16, 18): 0, (16, 19): 0, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 1, (17, 18): 1, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 0, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 0, (18, 21): 1,
        (19, 15): 0, (19, 16): 1, (19, 17): 1, (19, 18): 0, (19, 19): 1, (19, 20): 0, (19, 21): 1,
    },
    'HLC': { # Revisado
        (15, 15): 1, (15, 16): 0, (15, 17): 1, (15, 18): 0, (15, 19): 1, (15, 20): 1, (15, 21): 0,
        (16, 15): 1, (16, 16): 0, (16, 17): 0, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 0, (17, 16): 0, (17, 17): 1, (17, 18): 1, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 0, (18, 17): 1, (18, 18): 1, (18, 19): 0, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 0,
    },
    'MSB': { # Revisado
        (15, 15): 1, (15, 16): 1, (15, 17): 0, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 1, (16, 17): 1, (16, 18): 0, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 0, (17, 17): 1, (17, 18): 1, (17, 19): 0, (17, 20): 1, (17, 21): 1,
        (18, 15): 0, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 0, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 0, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 0,
    },
    'PMQ': { # Revisado
        (15, 15): 1, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 0,
        (16, 15): 1, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 0, (17, 18): 0, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 0, (18, 19): 0, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 0, (19, 19): 1, (19, 20): 0, (19, 21): 1,
    },
    'QWF': { # Revisado
        (15, 15): 0, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 1, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 1, (16, 17): 0, (16, 18): 1, (16, 19): 1, (16, 20): 0, (16, 21): 1,
        (17, 15): 0, (17, 16): 0, (17, 17): 1, (17, 18): 1, (17, 19): 0, (17, 20): 0, (17, 21): 1,
        (18, 15): 1, (18, 16): 0, (18, 17): 0, (18, 18): 0, (18, 19): 0, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 0, (19, 21): 1,
    },
    'EBB': { # Revisado
        (15, 15): 1, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 0,
        (16, 15): 1, (16, 16): 1, (16, 17): 0, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 0,
        (17, 15): 1, (17, 16): 1, (17, 17): 1, (17, 18): 0, (17, 19): 0, (17, 20): 1, (17, 21): 1,
        (18, 15): 0, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 0, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 0, (19, 20): 1, (19, 21): 0,
    },
    'IOE': { # Revisado
        (15, 15): 1, (15, 16): 0, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 0,
        (16, 15): 0, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 0, (17, 18): 0, (17, 19): 0, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 0,
        (19, 15): 1, (19, 16): 0, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 1,
    },
    'IOA': { # Revisado
        (15, 15): 1, (15, 16): 1, (15, 17): 0, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 1,
        (16, 15): 1, (16, 16): 0, (16, 17): 0, (16, 18): 0, (16, 19): 0, (16, 20): 0, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 0, (17, 18): 0, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 0, (18, 17): 0, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 0, (19, 19): 0, (19, 20): 0, (19, 21): 1,
    },
}

roles = {
  'RRD': ['P', 'S', 'V'],
  'QYV': ['P', 'S', 'V'],
  'LHL': ['P', 'V'],
  'HLC': ['S', 'V'],
  'MSB': ['P', 'S', 'V'],
  'PMQ': ['P', 'S', 'V'],
  'QWF': ['S', 'V'],
  'EBB': ['S', 'V'],
  'IOE': ['P', 'S', 'V'],
  'IOA': ['P', 'S', 'V'],
}

#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

La complejidad del problema viene dado por la cantidad de trinbunales que hay que asignar (n) y la cantidad de profesores disponibles y sus combinaciones (m).

**Orden de complejidad:** **O(m^n)**

Espacio de Soluciones incluiría todas las combinaciones posibles de 15 tribunales y 10 profesores<br>
El cálculo de las combinaciones posibles sería el siguiente:<br>
3 profesores distintos por tribunal: 10 x 9 x 8 = 720<br>
m = 720<br>
Considerando los 15 tribunales a asignar, hay que seleccionar entre esas 720 combinaciones 15 veces:<br>
**720^15 ≈ 7,2442e^42 soluciones posibles**, aunque con la poda no se exploran todas


#Diseño
- ¿Que técnica utilizo? ¿Por qué?

He empleado la técnica de **Ramificación y Poda**, ya que se trata una función objetivo para optimización y con restricciones específicas. Además, el problema trabaja solo con números enteros (1 y 0), lo cual encaja con esta técnica.
He procurado también usar la técnica de 'Divide y vencerás', aplicando recursividad para evitar bucles innecesarios.

# Resolución

- Preparación de datos

In [22]:
# Listas de profesores aptos para cada rol, para no tener que hacer la comprobación en cada iteración
presidentes = [ p for p in profesores if 'P' in roles[p] ]
secretarios = [ p for p in profesores if 'S' in roles[p] ]
vocales = [ p for p in profesores if 'V' in roles[p] ]

# Todos los turnos disponibles para consultar disponibilidad más tarde
turnos = [
  (15, 15), (15, 16), (15, 17), (15, 18), (15, 19), (15, 20), (15, 21), # día 15
  (16, 15), (16, 16), (16, 17), (16, 18), (16, 19), (16, 20), (16, 21), # día 16
  (17, 15), (17, 16), (17, 17), (17, 18), (17, 19), (17, 20), (17, 21), # día 17
  (18, 15), (18, 16), (18, 17), (18, 18), (18, 19), (18, 20), (18, 21), # día 18
  (19, 15), (19, 16), (19, 17), (19, 18), (19, 19), (19, 20), (19, 21)  # día 19
]

# Prefiltrado, donde hacemos una lista de profesores disponibles para cada turno
def revisar_disponibilidad():
  resultado = {}
  for turno in turnos:
      resultado[turno] = [ p for p in profesores if disponibilidad[p][turno] ]
  return resultado

# Crear diccionario con los profesores disponibes para cada día y hora
disp = revisar_disponibilidad()

- Inicialización de variables

In [23]:
mejor_solucion = None
cota_superior = float('inf') # desviación típica inicial de las cargas de trabajo de los profesores
solucion_actual = []
tribunales_ocupados = {}
carga_profesores = {p: 0 for p in profesores}
max = 15 # número total de tribunales a asignar

# Ordenar los turnos por cantidad de profesores disponibles, y seleccionar los 15 primeros turnos
turnos_por_disponibilidad = []
for slot in turnos:
  dia, hora = slot
  disponibles = len(disp[(dia, hora)])
  turnos_por_disponibilidad.append((slot, disponibles))

turnos_por_disponibilidad.sort(key=lambda x: x[1], reverse=True) # De mayor a menor

turnos_seleccionados = [turno for turno, _ in turnos_por_disponibilidad[:max]] # 15 turnos con mayor disponibilidad de profesores (usaremos para determinar la solución inicial)

- Algoritmo

In [34]:
# Función Objetivo
def calcular_desv_tipica(cargas = carga_profesores):
  total_profesores = len(cargas)
  valores = list(cargas.values())
  media = sum(valores) / total_profesores
  varianza = sum((v - media)**2 for v in valores) / total_profesores   # σ = √[Σ(xi - x̄)² / n]
  return varianza ** 0.5

def esta_ocupado(p, d, h):
  return (d, h) in tribunales_ocupados and p in tribunales_ocupados[(d, h)]

def asignar_tribunal(n, dia, hora, pres, sec, voc):
  solucion_actual.append({'tribunal': n + 1, 'dia': dia, 'hora': hora, 'P': pres, 'S': sec, 'V': voc})
  if (dia, hora) not in tribunales_ocupados:
    tribunales_ocupados[(dia, hora)] = []

  # Marcar qué profesores ocuparán el turno y sumar la asignación a cada profesor
  for prof in [pres, sec, voc]:
    tribunales_ocupados[(dia, hora)].append(prof)
    carga_profesores[prof] += 1

def rollback():
  if not solucion_actual:
    return
  t = solucion_actual.pop()

  # Desmarcar los profesores que ocupan el turno, y restar asignación
  for prof in [t['P'], t['S'], t['V']]:
    carga_profesores[prof] -= 1
    tribunales_ocupados[(t['dia'], t['hora'])].remove(prof)

  if not tribunales_ocupados[(t['dia'], t['hora'])]:
    del tribunales_ocupados[(t['dia'], t['hora'])]

def calcular_cota_inferior(idx):
  tribunales_asignados = idx
  tribunales_restantes = max - tribunales_asignados

  if tribunales_restantes == 0:
    return calcular_desv_tipica()

  # Simular la distribución de los tribunales restantes de forma optimista, sin tener en cuenta roles
  asignaciones_restantes = tribunales_restantes * 3
  cargas_simuladas = carga_profesores.copy() # copy para no modificar el original

  for _ in range(asignaciones_restantes):
    profesor_menos_cargado = min(cargas_simuladas, key=cargas_simuladas.get)
    cargas_simuladas[profesor_menos_cargado] += 1

  # Devolver la desviación típica de esta simulación para ver si merece la pena explorarla
  return calcular_desv_tipica(cargas_simuladas)

def generar_tribunales(idx_turno):
  global mejor_solucion, cota_superior

  # Condición de parada de cada rama
  if idx_turno >= max:
    desv = calcular_desv_tipica()

    # Actualizar cota superior si encontramos una mejor
    if desv < cota_superior:
      cota_superior = desv
      mejor_solucion = [t.copy() for t in solucion_actual]
    return

  # Creamos una rama y vemos si merece la pena explorar
  cota_inferior = calcular_cota_inferior(idx_turno)
  if cota_inferior >= cota_superior:
    # Podamos las cotas inferiores que no puedan mejorar la solución
    return

  #Si lleguamos aquí, exploramos la rama
  dia, hora = turnos_seleccionados[idx_turno]
  disponibles = set(disp[(dia, hora)])
  libres = {p for p in disponibles if not esta_ocupado(p, dia, hora)}

  # Verificar que haya al menos un profesor para cada rol. Ordenamos por carga
  prof_p = sorted(libres & set(presidentes), key=lambda p: carga_profesores[p])
  prof_s = sorted(libres & set(secretarios), key=lambda p: carga_profesores[p])
  prof_v = sorted(libres & set(vocales), key=lambda p: carga_profesores[p])

  if prof_p and prof_s and prof_v:
    # Asegurar que haya al menos un profesor distinto del resto para cada rol
    for p in prof_p:
      for s in prof_s:
        if s == p:
          continue
        for v in prof_v:
          if v in [p, s]:
            continue

          # Crear ramas con todas las combinaciones posibles
          asignar_tribunal(idx_turno, dia, hora, p, s, v)    # Asignar combinación
          generar_tribunales(idx_turno + 1)                  # Explorar siguiente tribunal
          rollback()                                         # Resetear asignación para explorar la siguiente combinación del bucle for

# Inicio del prodecimiento, empezando en el índice 0 (1er tribunal)
generar_tribunales(0)


print(f"Desviación típica final: {cota_superior:.4f}")

print("Contabilización de cargas para cada profesor")
cargas_finales = {p: 0 for p in profesores}
for t in mejor_solucion:
  cargas_finales[t['P']] += 1
  cargas_finales[t['S']] += 1
  cargas_finales[t['V']] += 1

print(cargas_finales)

print("\nTribunales:")
for t in mejor_solucion:
  print(f"  T{t['tribunal']}: Día {t['dia']}, Hora {t['hora']}:00 - P:{t['P']}, S:{t['S']}, V:{t['V']}")

Desviación típica final: 0.5000
Contabilización de cargas para cada profesor
{'RRD': 4, 'QYV': 4, 'LHL': 5, 'HLC': 5, 'MSB': 4, 'PMQ': 5, 'QWF': 4, 'EBB': 5, 'IOE': 4, 'IOA': 5}

Tribunales:
  T1: Día 19, Hora 17:00 - P:LHL, S:EBB, V:IOA
  T2: Día 15, Hora 18:00 - P:PMQ, S:RRD, V:QYV
  T3: Día 17, Hora 21:00 - P:MSB, S:HLC, V:IOE
  T4: Día 18, Hora 20:00 - P:IOA, S:QWF, V:EBB
  T5: Día 18, Hora 21:00 - P:LHL, S:PMQ, V:HLC
  T6: Día 15, Hora 17:00 - P:RRD, S:QYV, V:IOE
  T7: Día 16, Hora 15:00 - P:MSB, S:QWF, V:EBB
  T8: Día 16, Hora 19:00 - P:PMQ, S:HLC, V:RRD
  T9: Día 16, Hora 21:00 - P:LHL, S:IOA, V:MSB
  T10: Día 17, Hora 15:00 - P:QYV, S:IOE, V:EBB
  T11: Día 17, Hora 20:00 - P:LHL, S:IOA, V:PMQ
  T12: Día 18, Hora 17:00 - P:RRD, S:HLC, V:QYV
  T13: Día 18, Hora 18:00 - P:MSB, S:IOE, V:EBB
  T14: Día 19, Hora 15:00 - P:IOA, S:QWF, V:PMQ
  T15: Día 19, Hora 16:00 - P:LHL, S:QWF, V:HLC


# Uso de IA

La IA se usó inicialmente para generar el diccionario de disponibilidad de profesores, ya que era muy extenso (aunque hubo que revisarlo luego).<br>
Necesité consultar qué concepto matemático usar para medir el reparto de cargas para poder plantear la solución, de donde entendí que necesitaba calcular la desviación típica.<br>
También se usó para consultar si el planteamiento inicial con recursividad era coherente, y constultar posibles cambios de planteamiento cuando se me ocurrían y quería ver si era una mejora efectiva.<br>
Necesité confirmar las dudas surgidas durante la determinación de la complejidad.<br>